# 08 — Attention and Vision Transformers (ViT)

This notebook applies the Transformer architecture to images. Following the Vision Transformer idea, each image is split into **patches** that are linearly embedded, combined with positional embeddings and a class token, and processed by Transformer encoder blocks (multi-head self-attention, feed-forward network, skip connections and layer normalisation).

Two routes are planned, depending on computational budget: a **small ViT trained from scratch**, built step by step as in the course Transformer notebooks, and a **pretrained ViT / DeiT fine-tuned** on our classes. Attention maps are visualised to see which regions of the garment the model attends to.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Cobertura mínima: self-attention y Transformer

Implementar primero un ejemplo pequeño de consultas, claves, valores y softmax de puntuaciones; comprobar formas, suma de pesos por fila y efecto de cambiar una posición. Después aplicar un encoder Transformer pequeño a parches RGB64. Explicar qué añade respecto a una capa aislada de atención: posiciones, FFN, conexiones residuales y normalización. La variante ViT preentrenada es opcional; el ejemplo básico y su aplicación no dependen de una GPU potente. Los pesos de atención son evidencia del mecanismo, no una demostración causal de importancia. Pendiente de implementación.

## Puente entre arquitecturas: tokens, self-attention y Transformer

La pista «connection between tokens» se interpreta como las interacciones que modela self-attention: consultas, claves y valores derivados de la misma secuencia permiten combinar información entre posiciones. Cross-attention, en cambio, utiliza consultas de una secuencia y claves/valores de otra. Una atención sobre estados de una RNN es posible, pero un Transformer estándar no necesita recurrencia para establecer esas relaciones. Hay que aportar información de posición para representar el orden.

En ViT, los tokens son embeddings de parches de imagen; no palabras. Un mini-ViT scratch permite estudiar el mecanismo; un ViT preentrenado permite estudiar transferencia. Son rutas combinables y su viabilidad debe justificarse con el presupuesto local. Las ablaciones de parches o atención se diseñan para entender el mecanismo, no para asegurar que ViT gane a CNN.

**Corrección del apunte BERT → auto-regressive.** BERT original emplea un encoder bidireccional y predicción de tokens enmascarados; GPT usa una formulación causal de siguiente token. La flecha puede recoger un contraste tratado en clase; no prueba que el profesor identificara ambos. Ninguno es el modelo visual que vamos a entrenar aquí: sirven como ejemplos de cómo cambian máscara de atención y objetivo dentro de la familia Transformer. [Transformer](https://arxiv.org/abs/1706.03762), [BERT](https://arxiv.org/abs/1810.04805), [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf).


## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las doce clases elegidas tras los controles, no todo el catálogo.

### Resolución y presupuesto para la comparación

El mini-ViT puede usar RGB64 con parches 8×8 (64 tokens visuales, más el token de clase si se usa). El modelo preentrenado seguirá su entrada nativa documentada, por ejemplo 224×224. Mantener productos y splits, declarar número de tokens, dimensión, parámetros y coste. Las comparaciones a distinta resolución se etiquetan como aplicadas; la comparación controlada usa una resolución común. No atribuir a self-attention una ventaja observada junto con cambios de datos externos, capacidad y preprocesamiento.


## Objectives

- Implement patch embedding, positional encoding and a class token for H&M images.
- Build a small Transformer encoder (MHSA + FFN + skip + LayerNorm) and train it from scratch.
- Fine-tune a pretrained ViT / DeiT from `timm` or Hugging Face when resources allow.
- Study patch size, depth, number of heads and embedding dimension on a small budget.
- Visualise attention maps (e.g. attention rollout) on example garments.
- Discuss the data requirements of Transformers vs. the inductive bias of CNNs using our results.

## Preview of the work

1. **From images to tokens.** patching, linear projection, class token and positional embeddings.
2. **Mini-ViT from scratch.** encoder blocks reused from the course step-by-step notebooks.
3. **Training and learning curves.** comparison with CNN from scratch.
4. **Pretrained ViT fine-tuning.** if feasible on the available local hardware.
5. **Attention visualisation.** which parts of the garment drive the decision.
6. **Validation results and conclusions.** Transformers vs. CNNs on a mid-size dataset.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Loaders from notebook 02; results from notebooks 05 and 07 |
| **Outputs** | Metrics, checkpoints and attention-map figures in `results/` |
| **Course material** | `PRDL_MLLB_Attention_Mechanism_over_RNN.pptx`, `PRDL_MLLB_Transformer_Architecture_Part_I.pptx` and `Part_II.pptx`, `TODO_Notebook_01…03` (self-attention, MHSA, FFN + skip), `MLLB_HuggingFace_Intro.pptx` |